# AmsiNet - Step 8 (the planned "3b"): uncertainty-driven cascade, accuracy-versus-compute curve, learned negotiator

**Runtime:** Colab **CPU**. Needs the Step 3a probabilities, `results/backbone_efficiency.csv` (Step 3a/2 efficiency tables) and `agents/agent_efficiency_cpu.csv`. Nothing is trained on a GPU; everything is computed from saved probabilities, as in Step 6.

**What it tests**
1. **Bottom-up cascade.** The word agent (cheapest) runs on every word. A **line** agent is invoked only for lines containing a word whose uncertainty exceeds tau; a **block** agent only for blocks that still contain an uncertain word after the line stage. Uncertainty = normalised predictive entropy of the current posterior. Context from the other words in the line/block is free (their word predictions already exist).
2. **Accuracy versus compute.** For each tau: percentage of lines/blocks invoked, mean GFLOPs per image, test accuracy and macro-F1. Reference points: word agent only, all agents everywhere, and the flat ViT-B on every word.
3. **Operating point chosen on validation:** the cheapest tau that keeps at least 90% of the macro-F1 that the line and block agents add on top of the free word context; test is evaluated there with a paired bootstrap over images.
4. **Learned negotiator:** a small per-word gating model over the levels (word, line context, block context, line agent, block agent, page agent) with the mixture form of Eq. 16, fitted by cross-validation on the validation set (the agents' training-set probabilities are over-confident, so they cannot be used to fit it).

Costs count GFLOPs of the agents only. The word boxes come from the annotation, so the detector and the segmenter would add the same cost to every system and are left out. The page agent is not invoked (its fusion weight is 0 in Step 6).

In [ ]:
# 1. Setup
from google.colab import drive
import os
if not os.path.ismount('/content/drive'): drive.mount('/content/drive')
assert os.path.exists('/content/drive/MyDrive/amsinet/data/bstd_index.json'), 'Drive is not mounted or amsinet/data is missing: reconnect and run this cell again'
import json, numpy as np, pandas as pd
from scipy.optimize import minimize
from scipy.special import logsumexp, softmax
ROOT = '/content/drive/MyDrive/amsinet'
CFG = dict(WORD_TAG='word_moe_seed0', LINE_TAG='line_seed0', BLOCK_TAG='block_seed0', PAGE_TAG='page_seed0',
           BOOT=1000, SEED=0, EPS=1e-4, KEEP=0.90)

In [ ]:
# 2. Load index, crops, region table, probabilities, per-agent costs
D = json.load(open(f'{ROOT}/data/bstd_index.json', encoding='utf-8'))
SCRIPTS, INDEX = D['scripts'], D['index']; SIDX = {s: i for i, s in enumerate(SCRIPTS)}; C = len(SCRIPTS); IDX = {r['id']: r for r in INDEX}
META = pd.read_csv(f'{ROOT}/data/crops_meta.csv'); REGDF = pd.read_csv(f'{ROOT}/data/regions_meta.csv')
def load(tag, sp):
    P = np.load(f'{ROOT}/agents/{tag}_{sp}_probs.npy').astype(np.float32); pa = pd.read_csv(f'{ROOT}/agents/{tag}_{sp}_paths.csv'); return P, pa.path.tolist()
def region_lookup(level, tag, sp):
    P, paths = load(tag, sp); pos = {p: i for i, p in enumerate(paths)}
    d = REGDF[(REGDF.level == level) & (REGDF.split == sp)]
    return {(r.image_id, r.rid): P[pos[r.path]] for r in d.itertuples() if r.path in pos}

eff = pd.read_csv(f'{ROOT}/agents/agent_efficiency_cpu.csv').set_index('agent')['GFLOPs']
flat = pd.read_csv(f'{ROOT}/results/backbone_efficiency.csv'); flat = float(flat[flat.agent == 'Flat baseline']['GFLOPs'].iloc[0])
GF = dict(word=float(eff['word_moe'] if 'word_moe' in CFG['WORD_TAG'] else eff['word']), line=float(eff['line']), block=float(eff['block']), flat=flat)
print('GFLOPs per call:', GF)

## 3. Per-word table: word probabilities, context, region agents, group indices

In [ ]:
def table(sp):
    d = META[META.split == sp].reset_index(drop=True); n = len(d)
    Pw, pw = load(CFG['WORD_TAG'], sp); assert pw == d.path.tolist(), 'word probabilities are not aligned with crops_meta.csv'
    logP = np.log(np.clip(Pw, CFG['EPS'], 1)); lid, bid = [], []
    for iid, k in zip(d.image_id, d.word_idx):
        w = IDX[iid]['words'][k]; lid.append(w.get('line', -1)); bid.append(w.get('block', -1))
    img = d.image_id.values; lkey = img.astype(str) + '|l' + np.array(lid).astype(str); bkey = img.astype(str) + '|b' + np.array(bid).astype(str)
    lg, lu = pd.factorize(lkey); bg, bu = pd.factorize(bkey)
    def loo(g, ng):
        S = np.zeros((ng, C)); cnt = np.zeros(ng); np.add.at(S, g, logP); np.add.at(cnt, g, 1); o = cnt[g] - 1
        F = np.zeros((n, C)); m = o > 0; F[m] = (S[g][m] - logP[m]) / o[m][:, None]; return F
    def reg(level, tag, key):
        L = region_lookup(level, tag, sp); F = np.zeros((n, C))
        for i, (iid, kk) in enumerate(zip(img, key)):
            v = L.get((iid, f'{level[0]}{kk}'))
            if v is not None: F[i] = np.log(np.clip(v, CFG['EPS'], 1))
        return F
    PL = region_lookup('page', CFG['PAGE_TAG'], sp); Pa = np.zeros((n, C))
    for i, iid in enumerate(img):
        v = PL.get((iid, 'p'))
        if v is not None: Pa[i] = np.log(np.clip(v, CFG['EPS'], 1))
    # image index per group, to turn invoked-group counts into per-image costs
    ui, inv = np.unique(img, return_inverse=True); l_img = np.zeros(len(lu), int); l_img[lg] = inv; b_img = np.zeros(len(bu), int); b_img[bg] = inv
    return dict(y=np.array([SIDX[s] for s in d.script]), word=logP, line_ctx=loo(lg, len(lu)), block_ctx=loo(bg, len(bu)),
                line_agent=reg('line', CFG['LINE_TAG'], lid), block_agent=reg('block', CFG['BLOCK_TAG'], bid), page_agent=Pa,
                img=img, inv=inv, n_img=len(ui), lg=lg, bg=bg, nl=len(lu), nb=len(bu), l_img=l_img, b_img=b_img)
TAB = {sp: table(sp) for sp in ('val', 'test')}
TRAIN_PRIOR = np.bincount([SIDX[s] for s in META[META.split == 'train'].script], minlength=C).astype(float); TRAIN_PRIOR /= TRAIN_PRIOR.sum()
for sp, T in TAB.items(): print(sp, len(T['y']), 'words |', T['nl'], 'lines |', T['nb'], 'blocks |', T['n_img'], 'images')

In [ ]:
def confmat(y, p): return np.bincount(y * C + p, minlength=C * C).reshape(C, C)
def f1_from(cm):
    tp = np.diag(cm).astype(float); fp = cm.sum(0) - tp; fn = cm.sum(1) - tp; den = 2 * tp + fp + fn; return np.where(den > 0, 2 * tp / np.maximum(den, 1), 0.0)
def metr(y, p): cm = confmat(y, p); return dict(acc=float(np.trace(cm) / cm.sum()), f1=float(f1_from(cm).mean()))

# fusion weights for the log-linear combination, chosen on validation with ALL agents invoked (same procedure as Step 6, page agent excluded)
KEYS = ['line_ctx', 'block_ctx', 'line_agent', 'block_agent']; GRID_W = [0, .1, .25, .5, 1, 1.5, 2, 3]
def score_full(T, w): return T['word'] + sum(w[k] * T[k] for k in KEYS)
def val_f1(w): return metr(TAB['val']['y'], score_full(TAB['val'], w).argmax(1))['f1']
W = {k: 0.0 for k in KEYS}
for _ in range(2):
    for k in KEYS: W[k] = max(GRID_W, key=lambda g: val_f1({**W, k: g}))
print('fusion weights (validation):', W)
for sp in ('val', 'test'): print(sp, 'word only', metr(TAB[sp]['y'], TAB[sp]['word'].argmax(1)), '| all agents', metr(TAB[sp]['y'], score_full(TAB[sp], W).argmax(1)))

## 4. The cascade: invoke the line agent, then the block agent, only where uncertainty remains

In [ ]:
def unc(logits):                       # normalised predictive entropy of the current posterior, in [0, 1]
    p = softmax(logits, axis=1); return -(p * np.log(np.clip(p, 1e-12, 1))).sum(1) / np.log(C)

def cascade(T, tau, W=W):
    s = T['word'] + W['line_ctx'] * T['line_ctx'] + W['block_ctx'] * T['block_ctx']      # context is free
    u = unc(s)
    li = np.zeros(T['nl'], bool); np.logical_or.at(li, T['lg'], u > tau)                  # line invoked if any word in it is uncertain
    s = s + W['line_agent'] * T['line_agent'] * li[T['lg']][:, None]
    u = unc(s); bi = np.zeros(T['nb'], bool); np.logical_or.at(bi, T['bg'], u > tau)      # block invoked if uncertainty remains
    s = s + W['block_agent'] * T['block_agent'] * bi[T['bg']][:, None]
    return s, li, bi

def cost(T, li, bi, mode='cascade'):
    nw = np.bincount(T['inv'], minlength=T['n_img']).astype(float)
    nl = np.bincount(T['l_img'][li], minlength=T['n_img']).astype(float); nb = np.bincount(T['b_img'][bi], minlength=T['n_img']).astype(float)
    return float((nw * GF['word'] + nl * GF['line'] + nb * GF['block']).mean())

def point(T, tau):
    s, li, bi = cascade(T, tau); m = metr(T['y'], s.argmax(1)); return dict(tau=tau, lines_pct=100 * li.mean(), blocks_pct=100 * bi.mean(), gflops=cost(T, li, bi), acc=100 * m['acc'], f1=100 * m['f1'])

TAUS = [0, .02, .05, .1, .15, .2, .3, .4, .5, .6, .7, .8, .9, 1.01]
CUR = {sp: pd.DataFrame([point(TAB[sp], t) for t in TAUS]) for sp in ('val', 'test')}
nwo = lambda T: float((np.bincount(T['inv'], minlength=T['n_img']) * GF['flat']).mean())
REF = {sp: dict(word=dict(gflops=cost(TAB[sp], np.zeros(TAB[sp]['nl'], bool), np.zeros(TAB[sp]['nb'], bool)), **{k: 100 * v for k, v in metr(TAB[sp]['y'], TAB[sp]['word'].argmax(1)).items()}),
               flat_cost=nwo(TAB[sp])) for sp in ('val', 'test')}
print('tau=0 invokes everything (full fusion), tau>1 invokes nothing (word agent only + free context).'); display(CUR['test'].round(2))

## 5. Operating point chosen on validation, bootstrap on test, figure

In [ ]:
# cheapest tau on VALIDATION that keeps >= KEEP of the macro-F1 gain that the line/block agents add on top of free word context (conservative)
v = CUR['val']; f_word = 100 * metr(TAB['val']['y'], TAB['val']['word'].argmax(1))['f1']; f_full = float(v[v.tau == 0].f1.iloc[0]); f_ctx = float(v[v.tau > 1].f1.iloc[0])
target = f_ctx + CFG['KEEP'] * (f_full - f_ctx)
ok = v[v.f1 >= target]; TAU_STAR = float(ok.sort_values('gflops').tau.iloc[0]) if len(ok) else 0.0
print(f'validation macro-F1: word only {f_word:.2f} | context only (tau>1) {f_ctx:.2f} | full fusion {f_full:.2f} | target {target:.2f} -> tau* = {TAU_STAR}')

T = TAB['test']; y = T['y']; imgs = np.unique(T['img']); inv = T['inv']; ni = T['n_img']; rng = np.random.RandomState(CFG['SEED'])
def per_img(p):
    M = np.zeros((ni, C * C)); np.add.at(M, (inv, y * C + p), 1); return M
PRED = {'word agent only': T['word'].argmax(1), 'cascade at tau*': cascade(T, TAU_STAR)[0].argmax(1), 'all agents everywhere': cascade(T, 0)[0].argmax(1)}
CM = {k: per_img(p) for k, p in PRED.items()}; cnt = rng.multinomial(ni, np.ones(ni) / ni, size=CFG['BOOT']).astype(float)
def boot(k):
    cms = (cnt @ CM[k]).reshape(-1, C, C); acc = np.trace(cms, axis1=1, axis2=2) / cms.sum((1, 2)); tp = np.diagonal(cms, axis1=1, axis2=2); fp = cms.sum(1) - tp; fn = cms.sum(2) - tp
    den = 2 * tp + fp + fn; return acc, np.where(den > 0, 2 * tp / np.maximum(den, 1), 0).mean(1)
B = {k: boot(k) for k in PRED}; ci = lambda x: f'[{100 * np.percentile(x, 2.5):+.2f}, {100 * np.percentile(x, 97.5):+.2f}]'
star = point(T, TAU_STAR); full = point(T, 0)
rows = [('word agent only', REF['test']['word']['gflops'], 100 * metr(y, PRED['word agent only'])['acc'], 100 * metr(y, PRED['word agent only'])['f1'], '-', '-'),
        ('word agent + free line/block context (no extra agent calls)', point(T, 1.01)['gflops'], point(T, 1.01)['acc'], point(T, 1.01)['f1'], '-', '-'),
        (f'cascade (tau* = {TAU_STAR})', star['gflops'], star['acc'], star['f1'], ci(B['cascade at tau*'][0] - B['word agent only'][0]), ci(B['cascade at tau*'][1] - B['word agent only'][1])),
        ('all agents everywhere', full['gflops'], full['acc'], full['f1'], ci(B['all agents everywhere'][0] - B['word agent only'][0]), ci(B['all agents everywhere'][1] - B['word agent only'][1])),
        ('flat ViT-B/16 on every word (Step 2 test result)', REF['test']['flat_cost'], 93.07, 87.49, 'n/a', 'n/a')]
OP = pd.DataFrame(rows, columns=['system (BSTD test)', 'GFLOPs / image', 'accuracy %', 'macro-F1 %', 'delta acc vs word [95% CI]', 'delta F1 vs word [95% CI]']); display(OP.round(2))
da, df = B['cascade at tau*'][0] - B['all agents everywhere'][0], B['cascade at tau*'][1] - B['all agents everywhere'][1]
print('cascade vs all agents everywhere: delta acc %s | delta F1 %s | compute saved: %.1f%%' % (ci(da), ci(df), 100 * (1 - star['gflops'] / full['gflops'])))
OP.to_csv(f'{ROOT}/results/cascade_operating_point.csv', index=False); CUR['test'].to_csv(f'{ROOT}/results/cascade_curve_test.csv', index=False); CUR['val'].to_csv(f'{ROOT}/results/cascade_curve_val.csv', index=False)

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 2, figsize=(11, 4.2)); c = CUR['test']
for a, col, lab in ((ax[0], 'f1', 'macro-F1 (%)'), (ax[1], 'acc', 'accuracy (%)')):
    a.plot(c.gflops, c[col], '-o', ms=4, color='#2a6f97', label='AmsiNet cascade (sweep over tau)')
    a.scatter([REF['test']['word']['gflops']], [REF['test']['word']['f1' if col == 'f1' else 'acc']], marker='s', color='#888', zorder=3, label='word agent only')
    a.scatter([star['gflops']], [star[col]], marker='*', s=160, color='#d1495b', zorder=4, label='chosen tau* (validation)')
    a.scatter([REF['test']['flat_cost']], [87.49 if col == 'f1' else 93.07], marker='D', color='#333', zorder=3, label='flat ViT-B/16 on every word')
    a.set_xscale('log'); a.set_xlabel('GFLOPs per image (log scale)'); a.set_ylabel(lab); a.grid(alpha=.25)
ax[0].legend(fontsize=8, loc='lower right'); plt.suptitle('Accuracy versus compute on BSTD test (seed 0)'); plt.tight_layout()
plt.savefig(f'{ROOT}/results/fig_cascade_accuracy_vs_compute.png', dpi=200); plt.show()

## 6. Learned negotiator (Eq. 16, mixture form)
Each level gives a probability vector p_l (word, line context, block context, line agent, block agent, page agent). A small gating function produces one score per level from that level's own entropy and maximum probability plus a per-level bias, and the final prediction is the softmax-weighted **mixture** of the level distributions. It has 3 parameters per level (18 in total) and is fitted by 5-fold cross-validation on the validation set, grouped by image; the five fold models are averaged on test. This is the form of the negotiator that the data can support; a full Transformer encoder would need far more held-out probabilities than the validation set offers.

In [ ]:
LV = ['word', 'line_ctx', 'block_ctx', 'line_agent', 'block_agent', 'page_agent']
def level_probs(T): return np.stack([softmax(T[k], axis=1) for k in LV], 1)          # (n, L, C); zero features -> uniform = no information
def level_feats(P): return np.stack([-(P * np.log(np.clip(P, 1e-12, 1))).sum(2) / np.log(C), P.max(2)], 2)    # (n, L, 2)
def gate(theta, F):
    L = len(LV); b, a, c = theta[:L], theta[L:2 * L], theta[2 * L:]; return softmax(b[None] + a[None] * F[:, :, 0] + c[None] * F[:, :, 1], axis=1)
def nll(theta, P, F, y, l2=1e-3):
    w = gate(theta, F); pf = (w[:, :, None] * P).sum(1); return -np.log(np.clip(pf[np.arange(len(y)), y], 1e-9, 1)).mean() + l2 * (theta ** 2).sum()
def fit_gate(P, F, y): r = minimize(nll, np.zeros(3 * len(LV)), args=(P, F, y), method='L-BFGS-B', options=dict(maxiter=200)); return r.x
def predict_gate(theta, P, F): return ((gate(theta, F)[:, :, None] * P).sum(1))

Pv, Pt = level_probs(TAB['val']), level_probs(TAB['test']); Fv, Ft = level_feats(Pv), level_feats(Pt); yv = TAB['val']['y']
folds = np.unique(TAB['val']['img']); rs = np.random.RandomState(0); rs.shuffle(folds); fold_of = {im: i % 5 for i, im in enumerate(folds)}; fi = np.array([fold_of[i] for i in TAB['val']['img']])
thetas = []; oof = np.zeros((len(yv), C))
for k in range(5):
    tr, te = fi != k, fi == k; th = fit_gate(Pv[tr], Fv[tr], yv[tr]); thetas.append(th); oof[te] = predict_gate(th, Pv[te], Fv[te])
print('validation (out-of-fold) macro-F1: negotiator %.2f | log-linear fusion %.2f' % (100 * metr(yv, oof.argmax(1))['f1'], 100 * metr(yv, score_full(TAB['val'], W).argmax(1))['f1']))
Pn = np.mean([predict_gate(th, Pt, Ft) for th in thetas], 0); pn = Pn.argmax(1)
NEG = pd.DataFrame([('word agent only', *[100 * metr(y, PRED['word agent only'])[k] for k in ('acc', 'f1')]),
                    ('log-linear fusion (weights on val)', *[100 * metr(y, PRED['all agents everywhere'])[k] for k in ('acc', 'f1')]),
                    ('learned negotiator (Eq. 16 mixture, 5-fold on val)', *[100 * metr(y, pn)[k] for k in ('acc', 'f1')])], columns=['system (BSTD test)', 'accuracy %', 'macro-F1 %']); display(NEG.round(2))
w_mean = np.mean([gate(th, Ft).mean(0) for th in thetas], 0); print('mean fusion weight per level on test:', dict(zip(LV, w_mean.round(3))))
NEG.to_csv(f'{ROOT}/results/negotiator_vs_loglinear.csv', index=False)